# OMX Helsinki - yhtenäinen analyysidatasetti

Tämä notebook käyttää `src/stock_analysis`-pakettia, hakee jokaiselle tickerille yhden rivin ja tallentaa lopullisen datasetin vain kerran. Yksittäisen osakkeen hakuvika ei pysäytä koko ajoa eikä poista riviä datasetistä.

## 1. Asetukset

Muuta vuosia tässä solussa ennen ajoa.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from stock_analysis.config import AnalysisConfig
from stock_analysis.pipeline import run_analysis, save_dataset

CONFIG = AnalysisConfig(
    ticker_file=PROJECT_ROOT / 'Ticker_symbols.xlsx',
    max_workers=4,
    output_file=PROJECT_ROOT / 'stock_analysis_dataset.xlsx',
)
CONFIG

AnalysisConfig(ticker_file=PosixPath('/Users/timorautio/Desktop/stock_analysis/stock-analysis/Ticker_symbols.xlsx'), analysis_year=2025, comparison_year=2026, max_workers=4, output_file=PosixPath('/Users/timorautio/Desktop/stock_analysis/stock-analysis/stock_analysis_dataset.xlsx'))

## 2. Datan muodostaminen

Tämä on ajon ainoa verkkohakuosuus. Se voi kestää useita minuutteja.

In [2]:
dataset = run_analysis(CONFIG)
print(f'Rivejä: {len(dataset)}')
print(dataset['status'].value_counts(dropna=False))
dataset.head()

$AUROORA.HE: Data doesn't exist for startDate = 1735682400, endDate = 1767218400
$EASOR.HE: Data doesn't exist for startDate = 1735682400, endDate = 1767218400
$IQMX.HE: Data doesn't exist for startDate = 1735682400, endDate = 1767218400
$LASTIK.HE: Data doesn't exist for startDate = 1735682400, endDate = 1767218400
$REAKTOR.HE: Data doesn't exist for startDate = 1735682400, endDate = 1767218400
$SAVOX.HE: Data doesn't exist for startDate = 1735682400, endDate = 1767218400
$KPYOSK.HE: Data doesn't exist for startDate = 1735682400, endDate = 1767218400
$SBI.HE: Data doesn't exist for startDate = 1735682400, endDate = 1767218400


Rivejä: 194
status
ok    194
Name: count, dtype: int64


,symbol,yahoo_ticker,market_segment,name,analysis_year,comparison_year,status,error,total_revenue,eps,pb_ratio,pe_ratio,dividend_yield_pct,debt_to_equity,roe,analysis_year_last_close,comparison_year_last_close,price_change_pct_from_comparison
0,AALLON,AALLON.HE,First North GM,None,2025,2026,ok,None,3.963900e+07,0.61,2.465924,17.459016,2.159624,1.177725,0.140156,10.650000,9.000000,18.333329
1,ACG1V,ACG1V.HE,Main Market,None,2025,2026,ok,None,3.815000e+07,0.06,2.047811,84.666665,0.000000,0.539360,0.022170,5.080000,5.440000,-6.617649
2,ADMCM,ADMCM.HE,First North GM,None,2025,2026,ok,None,3.773578e+07,1.06,5.994975,40.660376,1.508121,0.169949,0.150915,43.099998,25.950001,66.088621
3,ADMIN,ADMIN.HE,First North GM,None,2025,2026,ok,None,7.330735e+07,-0.11,1.660998,-21.818183,2.083333,1.119190,-0.067569,2.400000,2.180000,10.091744
4,AFAGR,AFAGR.HE,Main Market,None,2025,2026,ok,None,1.412790e+08,-0.03,0.703789,-8.600000,0.000000,0.545530,-0.093221,0.258000,0.247000,4.453438


## 3. Laadun tarkistus ja suodatus

Kaikki lähdetiedoston tickerit säilyvät datasetissä. Tarkista ensin virherivit ja suodata sen jälkeen haluamasi osakkeet.

In [3]:
error_rows = dataset[dataset['status'].ne('ok')]
print(f'Virherivejä: {len(error_rows)}')
error_rows[['symbol', 'yahoo_ticker', 'error']].head(20)

Virherivejä: 0


,symbol,yahoo_ticker,error


In [4]:
# Esimerkki: yhtiöt, joiden ROE on positiivinen
filtered = dataset.query("status == 'ok' and roe > 0").sort_values('roe', ascending=False)
filtered[['symbol', 'name', 'roe', 'pe_ratio', 'pb_ratio']].head(20)

,symbol,name,roe,pe_ratio,pb_ratio
37,EAGLE,None,13.874529,-3.940000,-67.023332
96,LUOTEA,None,3.934307,0.572104,2.253798
66,HRTIS,None,3.922824,-7.285714,-32.038331
98,MARAS,None,1.749242,-0.960000,-1.688173
91,LEHTO,None,1.745233,-3.180000,-3.937753
49,FARON,None,1.472794,-9.423234,-22.801811
34,DOV1V,None,1.113516,-0.146800,-0.164363
136,REAKTOR,None,1.089548,NaN,NaN
146,SAVOX,None,0.546041,NaN,NaN
168,TEKOVA,None,0.521368,6.942929,3.619830


## 4. Lopullinen vienti

Vienti tehdään vain kerran. `Dataset` sisältää analyysirivit ja `Run_metadata` ajon parametrit.

In [5]:
output_path = save_dataset(dataset, CONFIG)
print(f'Tallennettu: {output_path}')

Tallennettu: /Users/timorautio/Desktop/stock_analysis/stock-analysis/stock_analysis_dataset.xlsx


In [6]:
dataset

,symbol,yahoo_ticker,market_segment,name,analysis_year,comparison_year,status,error,total_revenue,eps,pb_ratio,pe_ratio,dividend_yield_pct,debt_to_equity,roe,analysis_year_last_close,comparison_year_last_close,price_change_pct_from_comparison
0,AALLON,AALLON.HE,First North GM,None,2025,2026,ok,None,3.963900e+07,0.61,2.465924,17.459016,2.159624,1.177725,0.140156,10.650000,9.000000,18.333329
1,ACG1V,ACG1V.HE,Main Market,None,2025,2026,ok,None,3.815000e+07,0.06,2.047811,84.666665,0.000000,0.539360,0.022170,5.080000,5.440000,-6.617649
2,ADMCM,ADMCM.HE,First North GM,None,2025,2026,ok,None,3.773578e+07,1.06,5.994975,40.660376,1.508121,0.169949,0.150915,43.099998,25.950001,66.088621
3,ADMIN,ADMIN.HE,First North GM,None,2025,2026,ok,None,7.330735e+07,-0.11,1.660998,-21.818183,2.083333,1.119190,-0.067569,2.400000,2.180000,10.091744
4,AFAGR,AFAGR.HE,Main Market,None,2025,2026,ok,None,1.412790e+08,-0.03,0.703789,-8.600000,0.000000,0.545530,-0.093221,0.258000,0.247000,4.453438
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189,WETTERI,WETTERI.HE,Main Market,None,2025,2026,ok,None,4.340560e+08,0.02,0.763797,8.675000,0.000000,3.719537,0.116109,0.173500,0.167500,3.582088
190,WITTED,WITTED.HE,First North GM,None,2025,2026,ok,None,5.268171e+07,-0.05,1.578321,-28.199999,1.418440,0.769298,-0.051113,1.410000,1.625000,-13.230771
191,WRT1V,WRT1V.HE,Main Market,None,2025,2026,ok,None,6.914000e+09,1.06,6.216513,28.679245,1.447368,1.939299,0.217135,30.400000,28.459999,6.816587
192,WUF1V,WUF1V.HE,Main Market,None,2025,2026,ok,None,1.223260e+08,0.31,1.106397,12.838710,4.020100,1.466642,0.087127,3.980000,4.500000,-11.555555
